# AI Traffic Violation Detection — YOLO26 Training on Google Colab (T4 GPU)

### Step-by-Step Instructions:
1. **Select GPU Hardware Accelerator**:
   - Click **Runtime** in the top navigation bar.
   - Click **Change runtime type**.
   - Under *Hardware accelerator*, select **T4 GPU**.
   - Click **Save**.
2. **Upload Dataset**:
   - Create `datasets/merged.zip` on your local PC using `python scripts/zip_dataset.py`.
   - Upload `merged.zip` to your Google Drive root directory (`MyDrive/merged.zip`).
3. **Run All Cells Below** in order.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## 2. Install Latest Ultralytics with YOLO26 Support

In [ ]:
!pip install -U ultralytics
import ultralytics
ultralytics.checks()

## 3. Locate & Unzip Dataset from Google Drive

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Check standard locations in Google Drive for merged.zip
drive_candidates = [
    Path('/content/drive/MyDrive/merged.zip'),
    Path('/content/drive/MyDrive/traffic_dataset/merged.zip'),
    Path('/content/drive/MyDrive/datasets/merged.zip'),
]

zip_source = None
for cand in drive_candidates:
    if cand.exists():
        zip_source = cand
        break

if zip_source is None:
    # Recursive search if placed in a subfolder
    print("Searching Drive for merged.zip...")
    matches = list(Path('/content/drive/MyDrive').rglob('merged.zip'))
    if matches:
        zip_source = matches[0]

if zip_source is None or not zip_source.exists():
    raise FileNotFoundError("merged.zip not found in Google Drive! Please upload it to your Google Drive root (MyDrive/merged.zip).")

print(f"Found dataset at: {zip_source}")
extract_dest = Path('/content/datasets/merged')
extract_dest.mkdir(parents=True, exist_ok=True)

print(f"Extracting dataset to {extract_dest}...")
with zipfile.ZipFile(zip_source, 'r') as z:
    z.extractall(extract_dest)

# Verify data.yaml
yaml_path = extract_dest / 'data.yaml'
yaml_content = f"""# YOLO26 Traffic Detection Dataset
path: {extract_dest}
train: images/train
val: images/val

nc: 3
names:
  0: helmet
  1: no_helmet
  2: number_plate
"""
yaml_path.write_text(yaml_content.strip() + '\n')
print("Configured data.yaml:")
print(yaml_path.read_text())

train_imgs = len(list((extract_dest / 'images' / 'train').glob('*.*')))
val_imgs = len(list((extract_dest / 'images' / 'val').glob('*.*')))
print(f"Dataset ready: {train_imgs} train images, {val_imgs} val images.")

## 4. Train YOLO26 Nano (`yolo26n.pt`) Model
- Architecture: YOLO26 (NMS-free, ultra-low latency)
- Epochs: 40, Batch: 16, Imgsz: 640, Patience: 10
- Expected duration on T4 GPU: ~35–55 minutes

In [ ]:
from ultralytics import YOLO

# Initialize YOLO26 Nano base checkpoint
model = YOLO('yolo26n.pt')

# Start training
results = model.train(
    data='/content/datasets/merged/data.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project='/content/runs/train',
    name='yolo26n_traffic',
    save=True,
    plots=True,
    verbose=True,
)
print("Training complete!")

## 5. Evaluate & Export Results to Google Drive
Extracts `best.pt`, `results.csv`, `confusion_matrix.png`, and saves detailed per-class precision / recall / mAP metrics directly to `MyDrive/traffic_model_output/`.

In [ ]:
import json
import shutil
from pathlib import Path

# Prepare output folder on Google Drive
drive_output_dir = Path('/content/drive/MyDrive/traffic_model_output')
drive_output_dir.mkdir(parents=True, exist_ok=True)

run_dir = Path('/content/runs/train/yolo26n_traffic')
best_weights = run_dir / 'weights' / 'best.pt'

if best_weights.exists():
    shutil.copy(best_weights, drive_output_dir / 'best.pt')
    print(f"Copied best.pt -> {drive_output_dir / 'best.pt'}")
else:
    print("Warning: best.pt not found in weights/")

# Copy charts and training logs
for fname in ['results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png', 'F1_curve.png', 'PR_curve.png']:
    fpath = run_dir / fname
    if fpath.exists():
        shutil.copy(fpath, drive_output_dir / fname)
        print(f"Copied {fname} -> {drive_output_dir / fname}")

# Run full validation on best model
print("\nRunning final validation evaluation...")
best_model = YOLO(str(best_weights))
val_metrics = best_model.val(data='/content/datasets/merged/data.yaml', device=0)

# Extract overall and per-class metrics
eval_data = {
    "model": "yolo26n.pt",
    "overall": {
        "precision": round(float(val_metrics.results_dict.get('metrics/precision(B)', 0)), 4),
        "recall": round(float(val_metrics.results_dict.get('metrics/recall(B)', 0)), 4),
        "mAP50": round(float(val_metrics.results_dict.get('metrics/mAP50(B)', 0)), 4),
        "mAP50_95": round(float(val_metrics.results_dict.get('metrics/mAP50-95(B)', 0)), 4),
    },
    "per_class": {}
}

class_names = ['helmet', 'no_helmet', 'number_plate']
for i, cls_name in enumerate(class_names):
    try:
        eval_data["per_class"][cls_name] = {
            "precision": round(float(val_metrics.box.p[i]), 4),
            "recall": round(float(val_metrics.box.r[i]), 4),
            "mAP50": round(float(val_metrics.box.ap50[i]), 4),
            "mAP50_95": round(float(val_metrics.box.ap[i]), 4),
        }
    except Exception as e:
        print(f"Could not extract class {cls_name}: {e}")

eval_json_path = drive_output_dir / 'evaluation_results.json'
with open(eval_json_path, 'w') as f_json:
    json.dump(eval_data, f_json, indent=2)

print(f"\nSaved evaluation metrics to {eval_json_path}:")
print(json.dumps(eval_data, indent=2))
print("\nAll outputs successfully saved to Google Drive!")

## 6. (Optional) Train YOLO26 Small (`yolo26s.pt`)
Higher capacity and accuracy, ~2x slower training. Uncomment and run if Colab quota and time permits.

In [ ]:
# Optional: Train yolo26s.pt
# model_s = YOLO('yolo26s.pt')
# results_s = model_s.train(
#     data='/content/datasets/merged/data.yaml',
#     epochs=40,
#     imgsz=640,
#     batch=16,
#     patience=10,
#     device=0,
#     project='/content/runs/train',
#     name='yolo26s_traffic',
#     save=True,
#     plots=True,
# )
# print("YOLO26s Training complete!")